# Notebook 2 · Pré-processamento

Objetivo: converter os problemas encontrados no NB_1 em regras claras e reproduzíveis. Este notebook prepara os dados para seleção de variáveis e modelação. Não escolhe o modelo final.

Ordem: ler, validar, corrigir valores conhecidos, decidir linhas utilizáveis, construir variáveis, separar treino e validação no tempo, e só então ajustar imputação, codificação e escala.

## 1. Ler e verificar

Cada linha é uma compra. ID identifica a linha de submissão, customer_id identifica o cliente e repeat_purchase_90d é o alvo.

In [26]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data" / "Raw" / "train.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "Raw"
assert (DATA / "train.csv").exists(), "Executa a partir da raiz ou de notebooks/."

train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
clientes = pd.read_csv(DATA / "clientes.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
print({name: df.shape for name, df in {
    "train": train_raw, "test": test_raw, "clientes": clientes, "sample": sample
}.items()})

{'train': (6534, 39), 'test': (986, 38), 'clientes': (1143, 13), 'sample': (986, 2)}


In [27]:
TARGET = "repeat_purchase_90d"
assert TARGET in train_raw and TARGET not in test_raw
assert set(train_raw.columns) - set(test_raw.columns) == {TARGET}
assert set(test_raw.columns) - set(train_raw.columns) == set()
assert train_raw["ID"].is_unique and test_raw["ID"].is_unique
assert clientes["customer_id"].is_unique
assert sample["ID"].tolist() == test_raw["ID"].tolist()
assert sample.columns.tolist() == ["ID", TARGET]
assert train_raw[TARGET].isin([0, 1]).all()
print("Estrutura, IDs e ordem da submissão: OK")

Estrutura, IDs e ordem da submissão: OK


## 2. Corrigir defeitos conhecidos

Esta etapa converte os diagnósticos do NB_1 em regras de correção. Ainda não elimina linhas nem preenche todos os valores em falta. A função `clean_fixed` trabalha sobre uma cópia, preservando os dados originais, e aplica as mesmas regras ao treino e ao teste. São regras fixas: não aprendem estatísticas a partir do teste.

### 2.1 Converter a data para permitir operações temporais

**NB_1 — secções 2 e 7.1.** `purchase_date` estava guardada como texto e aparecia inicialmente como uma categórica com 1492 valores distintos. Existiam também 65 entradas sem data válida, nas linhas praticamente vazias.

**Decisão e motivo.** Converter a coluna para `datetime`, para ordenar compras e calcular intervalos.

**Aplicação no código.** `pd.to_datetime(..., errors="coerce")` converte os valores reconhecidos e representa entradas vazias ou inválidas como `NaT` (data em falta).

### 2.2 Reconstruir médias históricas corrompidas

**NB_1 — secção 3.2.** `prior_mean_purchase_value` tinha 16 valores negativos e 16 valores iguais a `1e9` no treino. A relação `prior_mean_purchase_value ≈ prior_total_spend / prior_purchase_count` verificava-se em 98,62% dos casos avaliáveis, com a tolerância usada nessa análise. Nas 16 médias negativas, a reconstrução era positiva em todos os casos; nas linhas com sentinela, a mediana reconstruída era 698,57.

**Decisão e motivo.** Reconstruir apenas as médias negativas ou superiores ou iguais a `1e8`, quando há compras anteriores e gasto histórico total não negativo. A informação da própria linha permite uma estimativa mais fundamentada do que preencher estes casos com uma mediana geral. O limiar de `1e8` permite detetar os sentinelas de `1e9` encontrados no NB_1; não é um limite universal para valores de compra.

**Aplicação no código.** `invalid_mean` marca as médias suspeitas e `valid_parts` exige `prior_purchase_count > 0` e `prior_total_spend >= 0`. A divisão calcula a média reconstruída, substituindo contagens zero por `NaN` para evitar divisão por zero. As linhas suspeitas com componentes utilizáveis recebem a reconstrução; as restantes recebem `NaN`. Médias que já estavam em falta ou que não satisfazem o critério de invalidade não são alteradas.


### 2.3 Marcar gastos de 365 dias inválidos como desconhecidos

**NB_1 — secções 3.2 e 3.3.** `spend_365d` tinha 15 valores negativos e 15 sentinelas de `1e9` no treino, além de incoerências nas relações entre gastos históricos. Ao contrário da média histórica, não foi demonstrada uma fórmula para recuperar o gasto exato desta janela.

**Decisão e motivo.** Substituir valores negativos ou superiores ou iguais a `1e8` por `NaN`. O gasto histórico total pode incluir compras com mais de 365 dias, e o gasto dos últimos 180 dias não permite determinar o restante período. Usar zero afirmaria que não houve gasto; `NaN` representa que o gasto é desconhecido.

**Aplicação no código.** `.mask(...)` substitui apenas os valores que cumprem uma das condições. O preenchimento posterior destes vazios será ajustado apenas com o conjunto de treino, na secção de imputação.


### 2.4 Marcar intervalos negativos como desconhecidos

**NB_1 — secção 3.2.** `days_since_previous_purchase` e `prior_mean_gap_days` tinham, cada uma, 16 valores negativos no treino, todos iguais a `-4`. Um intervalo negativo é incompatível com a definição destas variáveis. A repetição sugere um código artificial ou erro sistemático, mas não identifica a causa.

**Decisão e motivo.** Substituir qualquer valor negativo destas duas colunas por `NaN`. Não usamos o valor absoluto: não sabemos se `-4` deveria ser `4`. Também não usamos zero, porque isso indicaria um intervalo real de zero dias.

**Aplicação no código.** O ciclo percorre as duas colunas e usa `.mask(...lt(0))`. Uma eventual reconstrução a partir das datas exigiria confirmar que as compras anteriores necessárias estão disponíveis; esta versão não assume essa cobertura.

### 2.5 Manter as incoerências ainda sem correção justificada

**NB_1 — secção 3.3.** Foram encontradas diferenças na soma proposta para `Total a Pagar`, médias históricas fora do intervalo mínimo–máximo, gastos de 365 dias superiores ao gasto histórico total e casos de `Custo > Total Liquido`.

**Decisão e motivo.** Não forçar estas relações nesta função. Uma inconsistência não revela automaticamente qual a coluna errada, e algumas relações dependem do contexto: por exemplo, custo superior ao valor líquido pode corresponder a uma venda com prejuízo. Corrigir sem essa informação poderia substituir valores legítimos por valores inventados.

**Pendência.** Repetir as verificações de coerência após a limpeza para distinguir as violações resolvidas das que permanecem. Esta função não resolve todas as incoerências identificadas no NB_1.

In [28]:
def clean_fixed(frame):
    out = frame.copy()
    out["purchase_date"] = pd.to_datetime(out["purchase_date"], errors="coerce")

    col = "prior_mean_purchase_value"
    invalid_mean = out[col].lt(0) | out[col].ge(1e8)
    valid_parts = out["prior_purchase_count"].gt(0) & out["prior_total_spend"].ge(0)
    reconstructed = out["prior_total_spend"] / out["prior_purchase_count"].replace(0, np.nan)
    out.loc[invalid_mean & valid_parts, col] = reconstructed[invalid_mean & valid_parts]
    out.loc[invalid_mean & ~valid_parts, col] = np.nan

    out["spend_365d"] = out["spend_365d"].mask(
        out["spend_365d"].lt(0) | out["spend_365d"].ge(1e8)
    )
    for gap in ("days_since_previous_purchase", "prior_mean_gap_days"):
        out[gap] = out[gap].mask(out[gap].lt(0))
    return out

train_clean = clean_fixed(train_raw)
test_clean = clean_fixed(test_raw)
for name, before, after in (("train", train_raw, train_clean), ("test", test_raw, test_clean)):
    changed = {c: int((before[c].notna() & after[c].isna()).sum())
               for c in ("spend_365d", "days_since_previous_purchase", "prior_mean_gap_days")}
    repaired = int(((before["prior_mean_purchase_value"].lt(0) |
                     before["prior_mean_purchase_value"].ge(1e8)) &
                    after["prior_mean_purchase_value"].notna()).sum())
    print(name, "médias reconstruídas:", repaired, "| valores tornados em falta:", changed)

train médias reconstruídas: 32 | valores tornados em falta: {'spend_365d': 30, 'days_since_previous_purchase': 16, 'prior_mean_gap_days': 16}
test médias reconstruídas: 4 | valores tornados em falta: {'spend_365d': 4, 'days_since_previous_purchase': 2, 'prior_mean_gap_days': 2}


## 3. Escolher linhas de treino

Depois de corrigir os defeitos conhecidos, decidimos quais as compras utilizáveis para treino e avaliação temporal. A decisão distingue registos praticamente vazios de compras válidas com informação parcial; não elimina automaticamente todas as linhas com valores em falta.

### 3.1 Excluir os registos praticamente vazios

**NB_1 — secção 3.1.** As cinco colunas `Total a Pagar`, `Total Bruto`, `Total Liquido`, `Total Imp_` e `Custo` tinham 65 valores em falta cada, sempre nas mesmas 65 linhas. Essas linhas tinham 36 valores em falta por linha, num ficheiro com 39 colunas. A secção 7.1 confirmou também 65 datas ausentes ou não convertíveis, correspondentes a estes registos. Não se trata de uma compra à qual falta apenas um atributo: quase toda a descrição da compra está ausente.

**Decisão e motivo.** Excluir estas linhas do conjunto utilizável. Preencher quase toda a descrição de uma compra com valores típicos acrescentaria sobretudo informação artificial, e a ausência de data impede posicionar a compra na validação temporal. É uma escolha inicial fundamentada na qualidade das variáveis, não no valor do alvo. A taxa de recompra destas linhas foi descrita no NB_1, mas não é usada como critério de exclusão.

**Aplicação no código.** `purchase_totals` identifica os cinco valores principais e `no_purchase_info` marca as linhas onde todos estão em falta. O uso de `.all(axis=1)` significa «faltam os cinco nesta linha»; não significa «falta pelo menos um». Por isso, uma linha com apenas um total em falta não é excluída por este critério.

### 3.2 Exigir data válida e alvo conhecido

**NB_1 — secções 4 e 7.1.** O alvo do treino estava completo e continha apenas 0 e 1. A data permite ordenar as compras e separar passado e futuro, respeitando a estrutura temporal identificada no NB_1.

**Decisão e motivo.** Uma linha utilizável precisa de ter alguma informação nos cinco totais, uma data válida e um alvo conhecido. Sem data não pode entrar nesta avaliação temporal; sem alvo não pode servir para aprendizagem supervisionada nem para avaliar previsões.

**Aplicação no código.** `usable` combina as três condições com `&` («e»): `~no_purchase_info` («não tem os cinco totais vazios»), data não ausente e alvo não ausente. Assim, a regra também excluiria uma linha com data inválida mesmo que tivesse totais preenchidos. Nos dados atuais, os problemas coincidem nas mesmas 65 linhas. A validade binária do alvo já foi verificada na secção 1.

**Alcance da regra.** `usable` é uma máscara de seleção, com um valor verdadeiro ou falso por linha. Nesta célula, os registos ainda não são apagados de `train_clean`; a máscara é aplicada na construção de `X`, `y` e `dates` na secção seguinte. Manter a tabela completa permite voltar a consultar as linhas excluídas.

### 3.3 Conservar compras com histórico incompleto ou vazios pontuais

**NB_1 — secções 3.1 e 4.** 2605 das 6534 linhas (39,87%) tinham pelo menos um valor em falta. Parte desses vazios é estrutural: por exemplo, um cliente sem compras anteriores não tem intervalo desde a compra anterior nem média histórica. Outros vazios são falhas pontuais em colunas como transporte ou pagamento. As regras de disponibilidade das estatísticas históricas foram testadas no NB_1.

**Decisão e motivo.** Não eliminar estas compras apenas por terem vazios. Apagar todas as linhas incompletas perderia quase 40% do treino e retiraria casos de clientes com pouco histórico, que também precisam de previsões. 

**O que o código garante.** Não usa `dropna()` sobre todas as colunas nem exige que todas as features estejam preenchidas. Uma compra com data e alvo válidos e pelo menos um dos cinco totais disponível passa este filtro.

### 3.4 Preservar todas as linhas do teste

**NB_1 — secções 2 e 7.1.** O teste tinha 986 compras, com identificadores correspondentes aos da submissão e sem datas ausentes ou inválidas. O objetivo exige uma previsão para cada ID.

**Decisão e motivo.** Não aplicar ao teste o filtro de exclusão do treino. Mesmo uma compra de teste com informação parcial precisa de uma previsão; a preparação posterior deve lidar com os seus valores em falta.

**Aplicação no código.** Os `assert` verificam que o número de linhas do teste coincide com o da submissão e que todas as datas de teste são válidas. São verificações: se uma condição falhar, a execução para para permitir investigar o problema; não eliminam linhas. A correspondência exata e a ordem dos IDs já foram verificadas na secção 1.

In [29]:
purchase_totals = ["Total a Pagar", "Total Bruto", "Total Liquido", "Total Imp_", "Custo"]
no_purchase_info = train_clean[purchase_totals].isna().all(axis=1)
usable = ~no_purchase_info & train_clean["purchase_date"].notna() & train_clean[TARGET].notna()
print("Linhas excluídas do treino:", int((~usable).sum()))
print("Linhas de treino utilizáveis:", int(usable.sum()))
assert len(test_clean) == len(sample)
assert test_clean["purchase_date"].notna().all()

Linhas excluídas do treino: 65
Linhas de treino utilizáveis: 6469


## 4. Escolher a informação que o modelo vai receber

Depois de limpar os dados e identificar as compras utilizáveis, construímos a tabela de **features**: as variáveis que o modelo recebe para prever se o cliente volta a comprar em 90 dias. A função `make_features()` organiza essa informação; ainda não treina um modelo.

### Usar ou não o clientes.csv?

**O que descobrimos no NB_1 — secção 9.** Todas as compras têm uma ficha de cliente correspondente e a chave `customer_id` é única nessa ficha. Por isso, podemos juntar os ficheiros sem multiplicar compras. No entanto, `customer_type` está totalmente vazia e algumas colunas, como nomes e moradas, quase identificam cada cliente individualmente.

**Decisão.** Começar sem os atributos de `clientes.csv` e comparar mais tarde se acrescentá-los melhora as previsões. O NB_1 não demonstrou ainda esse benefício num modelo.

A função permite testar uma alternativa com apenas `customer_since_year` e `country_or_region`, que descrevem o ano de registo e a localização geral do cliente:

- `make_features(train_clean)`: usa apenas a informação já presente nas compras.

### Que colunas ficam fora das entradas do modelo?

A função cria uma cópia e retira as seguintes colunas dessa cópia:

| Coluna | Motivo da decisão |
|---|---|
| `ID` | Identifica a compra para a submissão, não é uma característica da compra. |
| `customer_id` | Identifica o cliente; nesta versão evitamos entregar o código diretamente ao modelo. O histórico de compras mantém-se. |
| `purchase_date` | O NB_1 mostrou que é necessária para ordenar e dividir os dados no tempo. Guardamo-la separadamente. |
| `repeat_purchase_90d` | É a resposta a prever, por isso não pode ser uma entrada do modelo. |
| `random_noise` | Fica fora da versão base com base na análise da secção 8 do NB_1. Esta escolha é provisória e pode ser confirmada em validação; o nome da coluna não prova que seja inútil. |

As colunas continuam nas tabelas limpas. Retirá-las das features não apaga os dados originais.

Por fim, a função uniformiza os vazios das colunas de texto como `np.nan`. Ainda não os preenche nem converte categorias em números: isso acontece na secção 6. Também não cria novos rácios ou estatísticas nesta versão.

### O que o código produz?

| Variável | Conteúdo |
|---|---|
| `X_all` | Features de todas as compras do treino, antes de aplicar o filtro da secção 3. |
| `X` | Features das 6469 compras utilizáveis. |
| `y` | Resposta real, 0 ou 1, dessas mesmas compras. |
| `dates` | Datas dessas mesmas compras, para dividir treino e validação no tempo. |
| `X_test` | Features das 986 compras do teste, para as quais teremos de prever a resposta. |

**Interpretação do resultado obtido.** A versão base tem **34 variáveis: 28 numéricas e 6 categóricas**. Este número não significa que todas sejam realmente importantes. O código confirma que treino e teste têm as mesmas colunas, na mesma ordem, e que todas as linhas de teste foram preservadas.


In [11]:
CLIENT_COLUMNS = ["customer_id", "customer_since_year", "country_or_region"]

def make_features(clean_frame, include_clientes=False):
    out = clean_frame.copy()
    if include_clientes:
        out = out.merge(clientes[CLIENT_COLUMNS], on="customer_id",
                        how="left", validate="m:1", sort=False)
        assert len(out) == len(clean_frame)
    X = out.drop(columns=["ID", "customer_id", "purchase_date", TARGET, "random_noise"],
                 errors="ignore")
    text_cols = X.select_dtypes(include=["str", "object", "category"]).columns
    for col in text_cols:
        X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
    return X

X_all = make_features(train_clean)
X_test = make_features(test_clean)
X = X_all.loc[usable].copy()
y = train_clean.loc[usable, TARGET].astype(int)
dates = train_clean.loc[usable, "purchase_date"]
assert X.columns.tolist() == X_test.columns.tolist()
assert len(X_test) == len(test_raw)
print("Variáveis de base:", X.shape[1], "| numéricas:",
      X.select_dtypes(include="number").shape[1],
      "| categóricas:", X.select_dtypes(exclude="number").shape[1])

Variáveis de base: 34 | numéricas: 28 | categóricas: 6
